**HW5 - GAN**

Problem 1

Build and train a Generative Adversarial Network (GAN) to generate handwritten digits from MNIST dataset. 

Data: The MNIST dataset contains 70,000 images of handwritten digits between 0 and 9 with 28×28 pixel size. Most deep learning libraries, such as Keras, provide access to the MNIST dataset.

In [ ]:
# Import libraries
import shutil
from google.colab import files
import os

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader, TensorDataset
from IPython.display import display, Image
import time
import copy
from datetime import datetime


In [ ]:
# Import dataset (MNIST)

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Data Load Start")

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

BUFFER_SIZE = len(train_dataset)
BATCH_SIZE = 64

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Data Ready (MNIST train set loaded)")

Architecture: Please build a GAN by combining the discriminator and generator models, such that the discriminator follows the generator in a sequential setting.

The first step is to define the discriminator model. The discriminator solves the binary classification problem by determining whether the input image is real or fake. Please build a discriminator by adding the layers to the Sequential model using the configuration below.

1. Conv2D
- Filters: 32
- Kernel size: (3,3)
- Activation: ReLU
2. MaxPooling2D
- Pool size: (2,2)
3. Conv2D
- Filters: 64
- Kernel size: (3,3)
- Activation: ReLU
4. MaxPooling2D
- Pool size: (2,2)
5. Flatten
6. Dense
- Units: 1
- Activation: Sigmoid

In [ ]:
# Discriminator Model Definition

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.model(x)
        return x

Second step involves building the generator model. The generator model is responsible for creating new fake but plausible images of handwritten digits. Build a generator by adding layers to the Sequential model using the configuration below.

1. Dense
- Units: 128*7*7
2. LeakyReLU
- Alpha: 0.2
3. Reshape
- Size: (7,7,128)
4. Conv2DTranspose
- Filters: 128
- Kernel size: (4,4)
- Strides: (2,2)
- Padding: Zero padding
5. LeakyReLU
- Alpha: 0.2
6. Conv2DTranspose
- Filters: 1
- Kernel size: (7,7)
- Strides: (2,2)
- Activation: tanh
- Padding: Zero padding

In [ ]:
# Generator Model Definition 

class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 128 * 7 * 7),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Unflatten(1, (128, 7, 7)),

            nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(128, 1, kernel_size=7, stride=2, padding=3, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Models testing 

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Models Test Start")

generator = Generator()

noise = torch.from_numpy(np.random.normal(size=(1, 100)).astype(np.float32))
generator.eval()
with torch.no_grad():
    generated_image = generator(noise)

image_to_plot = generated_image[0, 0, :, :].cpu().numpy()

plt.imshow(image_to_plot, cmap='gray')
plt.show()

discriminator = Discriminator()
decision = discriminator(generated_image)
print (decision)

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Models Test Finish")

Training: The GAN should be compiled with an ADAM optimizer to minimize the binary cross entropy. Fake samples are generated by the generator by using random noise as input, while the real images are randomly selected from the MNIST dataset. The combined real and fake images, along with their respective labels, are passed to the discriminator model for training.

In [ ]:
# Set up device and models 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
gen_optimizer = optim.Adam(generator.parameters(), lr=1e-4)
disc_optimizer = optim.Adam(discriminator.parameters(), lr=1e-4)

In [ ]:
# Single epoch train step

def train_step(real_images, noise_dim=100):
    batch_size = real_images.size(0)
    
    real_labels = torch.ones(batch_size, 1, device=device)
    fake_labels = torch.zeros(batch_size, 1, device=device)
    
    gen_optimizer.zero_grad()
    noise = torch.randn(batch_size, noise_dim, device=device)
    fake_images = generator(noise)
    fake_outputs = discriminator(fake_images)
    gen_loss = criterion(fake_outputs, real_labels)
    gen_loss.backward()
    gen_optimizer.step()
    
    disc_optimizer.zero_grad()
    real_outputs = discriminator(real_images)
    real_loss = criterion(real_outputs, real_labels)
    fake_outputs = discriminator(fake_images.detach())
    fake_loss = criterion(fake_outputs, fake_labels)
    disc_loss = real_loss + fake_loss
    disc_loss.backward()
    disc_optimizer.step()
    
    return gen_loss.item(), disc_loss.item()

In [ ]:
# Loops

import matplotlib.pyplot as plt
from torchvision.utils import save_image

def generate_and_save_images(model, epoch, test_input):
    os.makedirs('./runs', exist_ok=True)
    model.eval()
    with torch.no_grad():
        generated = model(test_input).cpu()
    save_image(generated, f'./runs/image_at_epoch_{epoch:04d}.png', normalize=True)
    model.train()

# PLOT LOSS CURVES
def plot_loss(curve, loss_name, output_dir="Plot JPGs"):
    os.makedirs(output_dir, exist_ok=True)

    plt.figure(figsize=(8, 5))
    epochs = np.arange(1, len(curve) + 1)
    plt.plot(epochs, curve, marker='o', linewidth=1.5, label=loss_name)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{loss_name} Curve")
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{loss_name.replace(' ', '_')}.jpg")
    plt.savefig(filename, dpi=150)
    plt.close()

def train(generator, discriminator, train_loader, epochs, noise_dim=100, save_every=15, checkpoint_dir='./checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    fixed_noise = torch.randn(16, noise_dim, device=device)

    gen_loss_history = []
    disc_loss_history = []

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        gen_loss_total = 0.0
        disc_loss_total = 0.0

        for real_images, _ in train_loader:
            real_images = real_images.to(device)
            gen_loss, disc_loss = train_step(real_images, noise_dim)
            gen_loss_total += gen_loss
            disc_loss_total += disc_loss

        avg_gen_loss = gen_loss_total / len(train_loader)
        avg_disc_loss = disc_loss_total / len(train_loader)

        gen_loss_history.append(avg_gen_loss)
        disc_loss_history.append(avg_disc_loss)

        generate_and_save_images(generator, epoch, fixed_noise)

        if epoch % save_every == 0:
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'gen_optimizer_state_dict': gen_optimizer.state_dict(),
                'disc_optimizer_state_dict': disc_optimizer.state_dict(),
            }, os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}.pth'))

        print(f'Epoch [{epoch}/{epochs}]  Gen Loss: {avg_gen_loss:.4f}  Disc Loss: {avg_disc_loss:.4f}  Time: {time.time()-start_time:.2f}s')

    # Save one dedicated snapshot from the final trained generator.
    generator.eval()
    with torch.no_grad():
        final_generated = generator(fixed_noise).cpu()
    save_image(final_generated, './runs/final_epoch_samples.png', normalize=True)
    generator.train()

    print("Training complete")

    plot_loss(gen_loss_history, "Generator Loss")
    plot_loss(disc_loss_history, "Discriminator Loss")

    print("Plots saved")
    print("Saved final generator snapshot: ./runs/final_epoch_samples.png")

    return gen_loss_history, disc_loss_history

In [ ]:
## EXPERIMENT 

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Run Start")

gen_loss_history, disc_loss_history = train(generator, discriminator, dataloader, epochs=100)

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Run Finish")


# Zip and download loss plots immediately after training
plot_zip_dir = "Plot JPGs"
if os.path.isdir(plot_zip_dir):
    shutil.make_archive("Plot_JPGs", "zip", plot_zip_dir)
    print("Created Plot_JPGs.zip")
    if files is not None:
        try:
            files.download("Plot_JPGs.zip")
        except Exception as e:
            print(f"Download skipped or failed: {e}")
else:
    print(f"Plot folder not found: {plot_zip_dir}")

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Loss plots exported")

Deliverables: Please plot the loss curves and upload at least five handwritten digit images generated by the generator in GAN after training the GAN for 100 epochs. Please make sure to submit your working code files along with the generated images and the plots.

In [ ]:
# Plot Loss Curves

plot_dir = "Plot JPGs"
loss_plot_paths = [
    os.path.join(plot_dir, "Generator_Loss.jpg"),
    os.path.join(plot_dir, "Discriminator_Loss.jpg"),
]

for plot_path in loss_plot_paths:
    if os.path.exists(plot_path):
        print(f"Displaying: {plot_path}")
        display(Image(filename=plot_path))
    else:
        print(f"Plot not found: {plot_path}")


In [ ]:
# Display the last epoch generated image

last_epoch_image_path = './runs/final_epoch_samples.png'
if os.path.exists(last_epoch_image_path):
    print(f"Displaying: {last_epoch_image_path}")
    display(Image(filename=last_epoch_image_path))
else:
    print(f"Last epoch image not found: {last_epoch_image_path}")


In [ ]:
# Generate 5 random images to show what the model can do after training

num_images = 5
noise_dim = 100
export_dir = './runs/export_samples'
os.makedirs(export_dir, exist_ok=True)

generator.eval()
with torch.no_grad():
    sample_noise = torch.randn(num_images, noise_dim, device=device)
    generated_images = generator(sample_noise)

# Analyze generated tensor schema: (batch, channels, height, width)
print(f"Generated tensor shape: {tuple(generated_images.shape)}")
print(f"Value range before denorm: [{generated_images.min().item():.3f}, {generated_images.max().item():.3f}]")

# Convert from tanh range [-1, 1] to display range [0, 1]
generated_images = (generated_images + 1) / 2.0
generated_images = generated_images.clamp(0, 1).cpu()

fig, axes = plt.subplots(1, num_images, figsize=(12, 3))
for i in range(num_images):
    axes[i].imshow(generated_images[i, 0], cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f"Sample {i+1}")

    sample_path = os.path.join(export_dir, f"generated_sample_{i+1}.png")
    save_image(generated_images[i], sample_path)

plt.tight_layout()
plt.show()

print(f"Saved {num_images} new samples to: {export_dir}")
print("Run the next cell to export/download the image bundle.")


In [ ]:
# Export image bundle (last epoch image + 5 newly generated images)

export_dir = './runs/export_samples'
os.makedirs(export_dir, exist_ok=True)

last_epoch_image_path = './runs/final_epoch_samples.png'
if os.path.exists(last_epoch_image_path):
    shutil.copy(last_epoch_image_path, os.path.join(export_dir, 'final_epoch_samples.png'))
    print('Included final_epoch_samples.png in export bundle')
else:
    print('final_epoch_samples.png not found; bundle will include only newly generated samples')

zip_base = './runs/gan_generated_images_bundle'
zip_path = shutil.make_archive(zip_base, 'zip', export_dir)
print(f"Created image bundle: {zip_path}")

if files is not None:
    try:
        files.download(zip_path)
    except Exception as e:
        print(f"Download skipped or failed: {e}")
